In [1]:
from pathlib import Path
import json

# Find the root directory, no matter where we are. 
def find_project_root(marker="README.md"):
    p = Path.cwd()
    while p != p.parent:
        if (p / marker).exists():
            return p
        p = p.parent
    raise RuntimeError("Project root not found")

PROJECT_ROOT = find_project_root()

path = PROJECT_ROOT / "data/processed/redirect_urls.json"
print(path)

/home/seancm/code/salary_prediction_project/data/processed/redirect_urls.json


In [2]:

with open(path, "r") as f:
    redirect_urls = json.load(f)

In [3]:
import pandas as pd 
urls = pd.DataFrame(pd.Series(redirect_urls), columns=["redirect_url"])

In [20]:
len(urls)

14290

In [27]:
urls = urls[urls["redirect_url"].str.contains("details")]

In [28]:
len(urls)

13885

In [45]:
import requests
from bs4 import BeautifulSoup
import json
import random

HEADERS = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64; rv:121.0) Gecko/20100101 Firefox/121.0",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-AU,en;q=0.5",
}

def get_full_description(url):
    try:
        response = requests.get(url, headers=HEADERS, timeout=15)
        response.raise_for_status()
        
        soup = BeautifulSoup(response.text, 'html.parser')
        scripts = soup.find_all('script', type='application/ld+json')
        
        for script in scripts:
            data = json.loads(script.string)
            if isinstance(data, list):
                data = data[0]
            if 'description' in data:
                return data.get('description')
        
        return None
    
    except Exception as e:
        print(f"Error: {e}")
        return None

In [ ]:
# Iterate through the data frame of URLs, and attach the description in the description column. 
import time
import random
import json
import os
from datetime import datetime

progress_file = "data/progress_tracking/scrape_progress.json"
save_interval = 50

# Resume from progress if it exists
start_index = 0
if os.path.exists(progress_file):
    with open(progress_file, "r") as f:
        progress = json.load(f)
    start_index = progress["last_processed"]
    print(f"Resuming from URL #{start_index + 1}")

total = len(urls)
start_time = datetime.now()

for i, (index, row) in enumerate(urls.iterrows()):
    if i < start_index:
        continue

    url = row["redirect_url"]
    description = get_full_description(url)
    urls.at[index, "description"] = description

    status = "ok" if description else "no description"
    print(f"[{i+1}/{total}] {status}: {url[:70]}")

    # Save checkpoint
    if (i + 1) % save_interval == 0:
        urls.to_csv("data/progress_tracking/urls_with_descriptions.csv", index=False)
        with open(progress_file, "w") as f:
            json.dump({"last_processed": i + 1, "timestamp": datetime.now().isoformat()}, f)

        elapsed = datetime.now() - start_time
        avg = elapsed.total_seconds() / (i + 1 - start_index)
        remaining = (total - i - 1) * avg / 3600
        print(f"Saved. {elapsed} elapsed, ~{remaining:.1f}h remaining")

    if i + 1 < total:
        time.sleep(random.uniform(2, 3))

# Final save
urls.to_csv("data/raw/urls_with_descriptions.csv", index=False)
with open(progress_file, "w") as f:
    json.dump({"last_processed": total, "timestamp": datetime.now().isoformat()}, f)
print(f"Done. {total} URLs processed.")

Sleeping for 14.99 seconds...
Sleeping for 10.06 seconds...
Sleeping for 12.23 seconds...
Sleeping for 13.14 seconds...
Sleeping for 9.99 seconds...
Sleeping for 13.00 seconds...
Sleeping for 7.20 seconds...
Sleeping for 13.63 seconds...
Sleeping for 14.37 seconds...
Sleeping for 12.20 seconds...
Sleeping for 5.56 seconds...


KeyboardInterrupt: 

In [ ]:
urls.iloc[0]["redirect_url"]

In [ ]:
# print response.text as nicely formatted html with linebreaks.
# Force print the whole thing without abreviating it.

print(BeautifulSoup(response.text, 'html.parser').prettify())

<!DOCTYPE html>
<html lang="en-AU">
 <head>
  <meta charset="utf-8"/>
  <title>
   Data Scientist - adzuna.com.au
  </title>
  <link href="//zunastatic-abf.kxcdn.com/css/dist/adzuna-6.0.65-vsn.css" rel="stylesheet"/>
  <meta content="width=device-width, initial-scale=1" name="viewport"/>
  <link href="https://zunastatic-abf.kxcdn.com/js/vendor/LAB.js" rel="prefetch"/>
  <link href="https://www.googleadservices.com/pagead/conversion.js" rel="prefetch"/>
  <link href="https://nd.demdex.net/" rel="preconnect"/>
  <meta content="We search thousands of job sites so that you don't have to. Discover job vacancies in your local area and across Australia now!" name="description">
   <script>
    (function(w,d,s,l,i){w[l]=w[l]||[];w[l].push({'gtm.start':
    new Date().getTime(),event:'gtm.js'});var f=d.getElementsByTagName(s)[0],
    j=d.createElement(s),dl=l!='dataLayer'?'&l='+l:'';j.async=true;j.src=
    'https://www.googletagmanager.com/gtm.js?id='+i+dl;f.parentNode.insertBefore(j,f);
    })

In [42]:
# Get the 'description' field from the html using tags, not the function
soup = BeautifulSoup(response.text, 'html.parser')
script = soup.find('script', type='application/ld+json')

# Extract the description field from the script
data = json.loads(script.string)
if isinstance(data, list):
    data = data[0]
print(data)

{'itemListElement': [{'position': 1, 'item': 'https://www.adzuna.com.au', '@type': 'ListItem', 'name': 'Jobs'}, {'position': 2, 'item': 'https://www.adzuna.com.au/queensland', '@type': 'ListItem', 'name': 'Queensland'}, {'position': 3, '@type': 'ListItem', 'item': 'https://www.adzuna.com.au/brisbane-region', 'name': 'Brisbane Region'}, {'position': 4, '@type': 'ListItem', 'item': 'https://www.adzuna.com.au/brisbane-region/it-jobs', 'name': 'IT Jobs'}, {'name': 'Data Scientist', 'position': 5, '@type': 'ListItem'}], '@context': 'https://schema.org', '@type': 'BreadcrumbList'}


'[\n   {\n      "itemListElement" : [\n         {\n            "position" : 1,\n            "item" : "https://www.adzuna.com.au",\n            "@type" : "ListItem",\n            "name" : "Jobs"\n         },\n         {\n            "position" : 2,\n            "item" : "https://www.adzuna.com.au/queensland",\n            "@type" : "ListItem",\n            "name" : "Queensland"\n         },\n         {\n            "position" : 3,\n            "@type" : "ListItem",\n            "item" : "https://www.adzuna.com.au/brisbane-region",\n            "name" : "Brisbane Region"\n         },\n         {\n            "position" : 4,\n            "@type" : "ListItem",\n            "item" : "https://www.adzuna.com.au/brisbane-region/it-jobs",\n            "name" : "IT Jobs"\n         },\n         {\n            "name" : "Data Scientist",\n            "position" : 5,\n            "@type" : "ListItem"\n         }\n      ],\n      "@context" : "https://schema.org",\n      "@type" : "BreadcrumbList"\n 